# Classification de l'Occupation des Sols (LULC)

Ce notebook classifie le territoire en 6 classes (Urbain, Forêt, Agriculture, etc.) en exploitant l'intelligence artificielle Prithvi.

In [ ]:
!pip install geemap earthengine-api scikit-learn rasterio terratorch torch matplotlib -q
import ee, geemap, torch, rasterio
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
from terratorch import BACKBONE_REGISTRY

ee.Initialize(project='geocongoai-api')

In [ ]:
roi = ee.Geometry.Rectangle([15.15, -4.85, 15.25, -4.75])
image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi).median().clip(roi)
geemap.ee_export_image(image.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']), 'input.tif', scale=30, region=roi)

In [ ]:
model = BACKBONE_REGISTRY.build("prithvi_eo_v2_300", num_frames=1, in_chans=6, pretrained=True).eval().to('cpu')
with rasterio.open('input.tif') as src: img = src.read().astype(np.float32) / 10000.0
with torch.no_grad():
    out = model(torch.from_numpy(img).unsqueeze(0))
    feats = out[0] if isinstance(out, list) else out
    
feats_np = feats[0, 1:].numpy()
kmeans = MiniBatchKMeans(n_clusters=6).fit(feats_np)
lc_labels = kmeans.labels_.reshape(int(np.sqrt(feats_np.shape[0])), -1)

plt.imshow(lc_labels, cmap='tab10')
plt.title("Occupation des Sols (LULC)")
plt.show()